# cmxflow: Building and Optimizing Cheminformatics Workflows

**cmxflow** is a Python framework for building composable cheminformatics pipelines that can be optimized via Bayesian optimization.

## Two Usage Modes

cmxflow is designed to work both as:

1. **An Agentic Tool** - via MCP (Model Context Protocol) server, allowing LLM agents to build and optimize workflows conversationally
2. **A Programmatic API** - for direct Python usage in scripts and notebooks

This tutorial covers the programmatic API, building toward an optimized virtual screening workflow.

## Demo Data: ABL1 from DUD-E

We'll use data from the wonderful [DUD-E benchmark](http://dude.docking.org/) (Database of Useful Decoys: Enhanced):

- **benchmark.csv**: 1000 molecules (28 actives, 972 decoys) for ABL1 kinase
- **crystal_ligand.sdf**: Co-crystallized ligand from the ABL1 structure
- **receptor.pdb**: ABL1 protein structure

## Block Types

cmxflow workflows are built from four types of blocks:

| Block Type | Purpose | Example |
|------------|---------|----------|
| **SourceBlock** | Read molecules from files | `MoleculeSourceBlock` |
| **Block** | Transform molecules (1:1 or N:M) | `PropertyFilterBlock`, `MoleculeSimilarityBlock` |
| **SinkBlock** | Write molecules to files | `MoleculeSinkBlock` |
| **ScoreBlock** | Compute optimization objective | `EnrichmentScoreBlock` |

Workflows must start with a SourceBlock and end with either a SinkBlock (for execution) or ScoreBlock (for optimization).

In [ ]:
# Core imports
from cmxflow import Workflow
from cmxflow.operators import (
    MoleculeSimilarityBlock,
    PropertyFilterBlock,
    PropertyHeadBlock,
    RDKitBlock,
)
from cmxflow.opt import Optimizer
from cmxflow.scores import EnrichmentScoreBlock
from cmxflow.sinks import MoleculeSinkBlock
from cmxflow.sources import MoleculeSourceBlock

## Building Your First Workflow

Let's build a simple workflow that:
1. Reads molecules from a CSV file
2. Computes molecular weight
3. Filters to keep molecules with MW between 200-500
4. Writes results to a file

In [ ]:
# Build a simple filter workflow
workflow = Workflow(name="Filter by MW")
workflow.add(
    MoleculeSourceBlock(),
    RDKitBlock("rdkit.Chem.Descriptors.MolWt"),
    PropertyFilterBlock(filters="200<MolWt<500"),
    MoleculeSinkBlock(),
)

# Visualize the workflow structure
print(workflow)

In [ ]:
# Execute the workflow
workflow("benchmark.csv", "filtered_output.csv")
print("Workflow complete! Check filtered_output.csv")

## Required Inputs

Some blocks require inputs at runtime - either files (like reference molecules) or text (like filter expressions). These can be set during block creation or dynamically before execution.

In [ ]:
# Build a workflow with required inputs
workflow = Workflow(name="Dynamic Filter")
workflow.add(
    MoleculeSourceBlock(),
    RDKitBlock("rdkit.Chem.Descriptors.MolWt"),
    PropertyFilterBlock(),  # No filter specified yet!
    MoleculeSinkBlock(),
)

# Check what inputs are required
workflow.check()
required = workflow.get_required_input()
print("Required inputs:", required)

# Set the filter dynamically
workflow.set_required_input({"2.text@filters": "MolWt>400"})
print("\nWorkflow ready to execute!")

## Available Operators

cmxflow includes many operator blocks for common cheminformatics tasks. Some examples include:

| Block | Purpose |
|-------|----------|
| `RDKitBlock` | Apply any RDKit method (descriptors, transformations) |
| `PropertyFilterBlock` | Filter molecules by property conditions |
| `PropertyHeadBlock` | Select top N molecules by property (highest) |
| `PropertyTailBlock` | Select bottom N molecules by property (lowest) |
| `MoleculeSimilarityBlock` | Compute 2D fingerprint similarity to references |
| `EnumerateStereoBlock` | Enumerate all stereoisomers |
| `ConformerGenerationBlock` | Generate 3D conformers (ETKDGv3) |
| `MoleculeAlignBlock` | Align molecules to 3D reference |
| `MoleculeDockBlock` | Dock into protein binding pocket |

## 2D Similarity Search

Let's build a similarity search workflow using the ABL1 crystal ligand as our reference. This finds molecules most similar to the known binder.

In [ ]:
# Build a 2D similarity search workflow
workflow = Workflow(name="2D Similarity Search")
workflow.add(
    MoleculeSourceBlock(),
    MoleculeSimilarityBlock(queries="crystal_ligand.sdf"),
    PropertyHeadBlock(property="max_similarity", count="100"),
    MoleculeSinkBlock(),
)

workflow

In [ ]:
# Execute: find top 100 most similar molecules
workflow("benchmark.csv", "similar_molecules.parquet")
print("Found top 100 similar molecules!")

## Mutable Parameters

Many blocks have **mutable parameters** that can be optimized. These come in three types:

- **Categorical**: Discrete choices (e.g., fingerprint type)
- **Integer**: Integer range (e.g., fingerprint radius)
- **Continuous**: Float range (e.g., similarity threshold)

Let's inspect the parameters on `MoleculeSimilarityBlock`:

In [ ]:
# Create a similarity block and inspect its parameters
sim_block = MoleculeSimilarityBlock(queries="crystal_ligand.sdf")

print("Mutable parameters:")
for name, param in sim_block.params.items():
    print(f"  {name}: {param.get()} (options: {param.options})")

## Parallel Execution

For compute-intensive blocks (conformer generation, docking, etc.), cmxflow provides parallel execution utilities:

In [ ]:
from cmxflow.utils.parallel import make_parallel

# Wrap any Block for parallel execution
sim_block = MoleculeSimilarityBlock(queries="crystal_ligand.sdf")
parallel_sim = make_parallel(
    sim_block,
    max_workers=4,      # Number of parallel processes
    ordered=True,       # Preserve input order
    error_handling="skip",  # Skip failed molecules
)

print("Created parallel block")
print("\nParallel blocks can be added to workflows just like regular blocks.")
parallel_sim

## Score Blocks for Optimization

To optimize a workflow, it must end with a **ScoreBlock** that computes a numeric objective:

| ScoreBlock | Purpose |
|------------|----------|
| `EnrichmentScoreBlock` | Enrichment AUC - how well does ranking separate actives? |
| `AverageScoreBlock` | Mean of a property (e.g., docking score) |
| `ShapeOverlayScoreBlock` | Average 3D shape similarity to references |

For virtual screening, **EnrichmentScoreBlock** is ideal:
- Score of **1.0** = perfect ranking (all actives ranked first)
- Score of **0.5** = random ranking
- Score of **0.0** = worst possible (all actives ranked last)

## Building an Optimizable Workflow

Now let's combine similarity search with enrichment scoring. The goal: find the fingerprint parameters that best separate ABL1 actives from decoys.

In [ ]:
# Build an optimizable similarity workflow
workflow = Workflow(name="Optimizable Similarity Search")
workflow.add(
    MoleculeSourceBlock(),
    MoleculeSimilarityBlock(queries="crystal_ligand.sdf"),
    EnrichmentScoreBlock(target="active"),  # Use 'active' column as labels
)

print(workflow)
print("\nThis workflow:")
print("1. Reads molecules from benchmark.csv")
print("2. Computes 2D similarity to crystal ligand")
print("3. Scores enrichment: how well does similarity rank actives?")

In [ ]:
# Check what parameters will be optimized
params = workflow.get_params()
print(f"Optimizable parameters ({len(params)} total):")
for p in params:
    print(f"  {p.name}: {p.get()} (options: {p.options})")

## Running Optimization

The `Optimizer` class uses Bayesian optimization (via Optuna) to find the best parameters:

In [ ]:
# Create optimizer
optimizer = Optimizer(workflow, "benchmark.csv")

# Run optimization (30 trials is usually sufficient for this search space)
study = optimizer.optimize(
    n_trials=30,
    direction="maximize",  # Maximize enrichment AUC
    show_progress_bar=True,
)

In [ ]:
# Show optimization results
print(f"Best enrichment AUC: {optimizer.best_score:.3f}")
print("\nBest parameters:")
for name, value in optimizer.best_params.items():
    print(f"  {name}: {value}")

In [ ]:
# Apply best parameters to the workflow
optimizer.set_best_params()
print("Best parameters applied to workflow!")

# Verify parameters were updated
workflow

## Analyzing Optimization Results

The Optuna study object provides rich analysis capabilities:

In [ ]:
import optuna

# Plot optimization history
fig = optuna.visualization.plot_optimization_history(optimizer.study)
fig.show()

In [ ]:
# Plot parameter importance
fig = optuna.visualization.plot_param_importances(optimizer.study)
fig.show()

## Agentic Usage

cmxflow includes an MCP (Model Context Protocol) server that allows LLM agents to build and optimize workflows conversationally.

**Three main tools:**
- `build_workflow`: Create workflows step-by-step (create, add_block, validate, etc.)
- `run_workflow`: Execute validated workflows (get_inputs, set_inputs, execute)
- `optimize_workflow`: Run Bayesian optimization (start, status, get_best_params)

**Example agent interaction:**
```
User: Build a 2D similarity workflow using morgan fingerprints
Agent: [calls build_workflow with appropriate blocks]

User: Optimize it on benchmark.csv to maximize enrichment
Agent: [calls optimize_workflow with n_trials=30, direction="maximize"]

User: Show me the best parameters
Agent: [calls optimize_workflow with action="get_best_params"]
```

The MCP server maintains workflow state across calls, enabling natural conversational workflow building.

## Saving and Loading Workflows

Workflows can be serialized to disk for later use. This preserves all block configurations, parameter values, and required inputs.

In [ ]:

# Save the optimized workflow
# save_workflow(workflow, "optimized_similarity_workflow.pkl")
# print("Workflow saved to optimized_similarity_workflow.pkl")

# Load it back later
# loaded_workflow = load_workflow("optimized_similarity_workflow.pkl")
# print(f"Loaded workflow: {loaded_workflow.name}")

# The loaded workflow retains all optimized parameters!
# for p in loaded_workflow.get_params():
#     print(f"  {p.name}: {p.get()}")

print("save_workflow(workflow, 'path.pkl')  - Serialize workflow to file")
print("load_workflow('path.pkl')            - Load workflow from file")
print("\nLoaded workflows retain all parameters and configurations!")

## Summary

In this tutorial, we covered:

1. **Block Types**: Source, Block, Sink, and Score blocks form the workflow building blocks
2. **Workflow Construction**: Chain blocks together with `workflow.add()`
3. **Required Inputs**: Dynamic file and text inputs via `set_required_input()`
4. **Mutable Parameters**: Categorical, Integer, and Continuous parameters for optimization
5. **Parallel Execution**: `make_parallel()` for compute-intensive blocks
6. **Optimization**: Bayesian optimization with `Optimizer` to find best parameters
7. **Persistence**: `save_workflow()` and `load_workflow()` for serialization

### Key Takeaways

- cmxflow makes cheminformatics pipelines composable and optimizable
- The same workflows work both programmatically and via the agentic MCP interface
- Bayesian optimization efficiently searches parameter spaces
- DUD-E benchmarks (like ABL1) are excellent for validating virtual screening workflows

### Next Steps

- Try 3D workflows with `ConformerGenerationBlock` and `MoleculeAlignBlock`
- Explore docking with `MoleculeDockBlock` and `receptor.pdb`
- Build custom blocks by subclassing `MoleculeBlock`